# avgpool-reduce — ex1: build AvgPool2d via einops.reduce

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `avgpool-reduce`. Running the final beacon cell reports progress against the `CNN: AvgPool as reduce` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: AvgPool as reduce` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`avgpool-reduce`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "avgpool-reduce"
DD_SUBTOPIC = "CNN: AvgPool as reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## AvgPool2d == einops.reduce(mean) — quick refresher

`nn.AvgPool2d(p)` with non-overlapping windows is exactly an einops `reduce` with the `mean` op:

```
y = einops.reduce(x, 'b c (h p1) (w p2) -> b c h w', 'mean', p1=p, p2=p)
```

**Reading the einops string.**
- `(h p1)` factors the input H-axis into `h * p1` — `h` is the output spatial axis, `p1` is the pool-window axis being reduced.
- Same trick on width: `(w p2)`.
- Right-hand side keeps `b c h w` — the pool axes `p1, p2` are dropped (that's the reduction).
- `'mean'` averages over the dropped axes.

**Global avg-pool** is the all-spatial special case — collapse the ENTIRE `(H, W)` into one scalar per (batch, channel):

```
y = einops.reduce(x, 'b c h w -> b c', 'mean')
```

This is exactly what ResNet does between the last conv block and the classifier — turn `(B, 512, 7, 7)` into `(B, 512)` so a `Linear` can consume it. Equivalent to `nn.AdaptiveAvgPool2d(1)` followed by `squeeze(-1).squeeze(-1)`.

**Why einops is cleaner.** No need to compute `kernel_size` or `stride` explicitly — the (h p1) factor pattern declares both at once. And it trivially extends to 3-D / N-D pooling.

### Exercise 1 — build AvgPool2d via einops.reduce

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `einops.reduce('mean')` with axis-factoring to reproduce `nn.AvgPool2d` with non-overlapping windows, and verify against `F.avg_pool2d`.
> Keywords: avgpool, einops-reduce, global-pool, resnet-head
> ```

**KCs targeted:** `avgpool-as-reduce-mean`, `global-avgpool-collapse`

Implement `ex1_avgpool_via_reduce(x, p)`. Given input `x: (B, C, H, W)` and pool size `p` (assume `H` and `W` are divisible by `p`), return a `(B, C, H // p, W // p)` tensor whose entries are the **mean** of each non-overlapping `p × p` window of `x`.

**Use einops.reduce with axis factoring.** The pattern is:

```
einops.reduce(x, 'b c (h p1) (w p2) -> b c h w', 'mean', p1=p, p2=p)
```

**Read it letter-by-letter.**
- `(h p1)` says 'factor the input H axis into `h * p1`' — `h` will appear on the output as the pooled axis; `p1` is dropped (reduced).
- `'mean'` averages over the dropped axes.
- Pass `p1=p, p2=p` so einops knows the factor sizes.

**Boundary handling.** This drill assumes `H % p == 0` and `W % p == 0`. Real `nn.AvgPool2d` can pad to handle non-divisible sizes — out of scope here.

The test compares your output to `F.avg_pool2d(x, kernel_size=p)` to fp tolerance.

In [ ]:
def ex1_avgpool_via_reduce(x: Tensor, p: int) -> Tensor:
    """Non-overlapping AvgPool2d via einops.reduce mean."""
    raise NotImplementedError()


def _test_ex1():
    from torch.nn import functional as F

    rng = t.Generator().manual_seed(0)

    # Small hand-checkable case.
    x = t.tensor([
        [[[1.0, 2.0, 3.0, 4.0],
          [5.0, 6.0, 7.0, 8.0],
          [9.0, 1.0, 2.0, 3.0],
          [4.0, 5.0, 6.0, 7.0]]]
    ])  # shape (1, 1, 4, 4)
    y = ex1_avgpool_via_reduce(x, p=2)
    assert y.shape == (1, 1, 2, 2), f'expected (1,1,2,2), got {tuple(y.shape)}'
    expected = t.tensor([[[
        [(1+2+5+6)/4, (3+4+7+8)/4],
        [(9+1+4+5)/4, (2+3+6+7)/4],
    ]]])
    assert t.allclose(y, expected, atol=1e-6), f'value mismatch:\n{y}\nvs\n{expected}'

    # Cross-check against F.avg_pool2d on random data, various sizes.
    for B, C, H, W, p in [(2, 3, 8, 8, 2), (1, 4, 16, 16, 4), (3, 2, 12, 6, 2), (1, 1, 32, 32, 8)]:
        xr = t.randn(B, C, H, W, generator=rng)
        yr = ex1_avgpool_via_reduce(xr, p)
        yref = F.avg_pool2d(xr, kernel_size=p)
        assert yr.shape == yref.shape
        assert t.allclose(yr, yref, atol=1e-5), (
            f'mismatch for shape ({B},{C},{H},{W}) p={p}'
        )

    # p == 1 is the identity (each window is a single pixel).
    x_id = t.randn(1, 2, 4, 4, generator=rng)
    assert t.allclose(ex1_avgpool_via_reduce(x_id, 1), x_id, atol=1e-7)

    # Constant input → constant output (mean of constants = constant).
    x_c = t.full((2, 3, 6, 6), 4.2)
    y_c = ex1_avgpool_via_reduce(x_c, 3)
    assert y_c.shape == (2, 3, 2, 2)
    assert t.allclose(y_c, t.full((2, 3, 2, 2), 4.2), atol=1e-6)
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_avgpool_via_reduce(x: Tensor, p: int) -> Tensor:
    return einops.reduce(
        x,
        'b c (h p1) (w p2) -> b c h w',
        'mean',
        p1=p, p2=p,
    )
```

**Why `(h p1)` and not `(p1 h)`.** Order inside the parentheses matters for einops factoring. `(h p1)` means 'rows are grouped in BLOCKS of size `p1` — `h` is the block index'. This is the non-overlapping-window semantics that matches AvgPool. `(p1 h)` would interleave (stride-style) and produce a different tensor — not what we want.

**Global avg-pool variant.** Collapse the whole spatial extent: `einops.reduce(x, 'b c h w -> b c', 'mean')`. This is what ResNet does between the last BlockGroup and the classifier — turns `(B, 512, 7, 7)` into `(B, 512)` so `nn.Linear` can consume it.

**Equivalence with adaptive pool.** `nn.AdaptiveAvgPool2d((1, 1))(x).squeeze(-1).squeeze(-1)` does the global-pool version. Both produce identical output; the einops form is more transparent about what's happening.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()